<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/6_tft_model/6_2_coarse_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **6_2_coarse_tuning**

**Objetivo del tuneo grueso (coarse tuning)**

El tuneo grueso busca:

- Identificar órdenes de magnitud correctos
- Definir una arquitectura razonable
- Descartar regiones malas del espacio de hiperparámetros
- Llegar a un baseline optimizado, sobre el cual luego hacer fine tuning

**No buscamos el mejor modelo, sino el mejor espacio.**

## **Introducción y Resumen**

Temporal Fusion Transformer (TFT) fue propuesto por Google (Lim et al., 2020).
Está diseñado específicamente para series temporales multivariadas y combina lo mejor de varios mundos:

- LSTM → para capturar dependencias temporales locales.

- Self-Attention (Transformer) → para capturar relaciones de largo plazo y entre features.

- Gating + Variable Selection Networks → para seleccionar dinámicamente qué features son más relevantes en cada instante.

- Interpretabilidad → puedes visualizar la importancia temporal y por variable.

👉 En tu caso:

- Tienes ventanas fijas de 60 minutos (window_size=60).
- Cada ventana tiene muchas features (entre 900 y 1080), con relaciones complejas.
- Necesitas capturar patrones secuenciales y relevancia entre indicadores técnicos y alpha factors.

➡️ El TFT es ideal.


## **0. Configuración del Entorno**


### 0.1. Instalación de librerías


In [1]:
# ==============================================
# 1) ELIMINAR TODO LO VIEJO
# ==============================================
!pip uninstall -q -y torch torchvision torchaudio xformers lightning pytorch-forecasting pytorch-lightning

# ==============================================
# 2) INSTALAR PYTORCH GPU (CUDA 12.1) + TORCHVISION + TORCHAUDIO
# Compatible con Colab + Python 3.12
# ==============================================
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
torch==2.5.1

# ==============================================
# 3) INSTALAR LIGHTNING MODERNO + PYTORCH FORECASTING MODERNO
# (Compatibles con Python 3.12 y con el nuevo Lightning)
# ==============================================
!pip install -q "lightning>=2.2.0" "pytorch-forecasting"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 330.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 242.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 419.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 372.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 285.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 363.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 376.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 380.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 370.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 372.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 250.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 306.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

### 0.2. Importación de librerías


In [2]:
# -------------------------------------------------
# Configuración global (UNA SOLA VEZ)
# -------------------------------------------------
import torch
import logging

# Aprovechar Tensor Cores (GPU NVIDIA L4)
torch.set_float32_matmul_precision("high")  # o "medium"

# Reducir logs INFO de Lightning
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)

In [3]:
# ==============================
# Librerías de modelado (Lightning moderno)
# ==============================
import lightning.pytorch as pl
import torch

# ==============================
# PyTorch Forecasting (TFT y utilidades)
# ==============================
from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer,
)
from pytorch_forecasting.metrics import RMSE
from torch.utils.data import DataLoader

# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning tradicional
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

import joblib

warnings.filterwarnings("ignore")

# ===============================
# MODO SEGURO: SIN WIDGETS
# ===============================

# tqdm seguro (NO notebook widgets)
from tqdm.auto import tqdm

# Forzar backend matplotlib no interactivo
import matplotlib
matplotlib.use("Agg")

# Evitar render interactivo automático
import matplotlib.pyplot as plt
plt.ioff()

print("Modo sin widgets activado.")

Modo sin widgets activado.


In [4]:
import sys, platform
import numpy
import scipy
import sklearn
import torch
import lightning.pytorch as pl
import pytorch_forecasting

print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("Torch:", torch.__version__)
print("Lightning:", pl.__version__)
print("Forecasting:", pytorch_forecasting.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1
Torch: 2.5.1+cu121
Lightning: 2.6.0
Forecasting: 1.5.0


### 0.3. Acceso a Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.4. Comprobación de uso de RAM

In [6]:
import psutil

def ram_usage():
    ram = psutil.virtual_memory()
    used = ram.used / (1024**3)
    free = ram.available / (1024**3)
    total = ram.total / (1024**3)

    print(f"RAM total:      {total:.2f} GB")
    print(f"RAM usada:      {used:.2f} GB")
    print(f"RAM disponible: {free:.2f} GB")

In [7]:
ram_usage()

RAM total:      52.96 GB
RAM usada:      1.79 GB
RAM disponible: 50.55 GB


## **1. Carga de datos**

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [8]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [9]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [10]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [11]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [12]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_90 = features_dict["features_to_90"]


In [13]:
print(f'Listado de features para 90min ({len(features_to_90)}): {features_to_90}')

Listado de features para 90min (7): ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## **2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`**

In [14]:
#Lista de K folds
k_folds = [1, 2, 3, 4, 5]

features_base = ['open', 'high', 'low', 'close', 'volume']

features_90 = features_base + features_to_90

window_size = 90

### 2.0. Funciones

#### Función para cargar ventanas

In [15]:
def load_windows_and_scaler(k: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz'
    path_valid  = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz'
    path_test   = f'{drive_path}/5_transformer_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/5_transformer_model/5_2_k_scaler/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    print(f'\tX_train_sc_{k} e y_train_{k} extraídos correctamente')
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    print(f'\tX_valid_sc_{k} e y_valid_{k} extraídos correctamente')
    X_test,  y_test  = data_test["X"],  data_test["y"]
    print(f'\tX_test_sc_{k} e y_test_{k} extraídos correctamente')

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler

#### Función para revisar información de ventanas

In [16]:
def xy_info(k, X_train, y_train, X_valid, y_valid, X_test, y_test, silent=False):
    import numpy as np
    import psutil

    if not silent:
        print(f"Información de {k}:")
        print("----------------------------------------")

    # Memoria total
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)

    def print_set_info(nombre, X, y):
        if silent:
            return  # No imprimir nada

        n_samples = X.shape[0]
        size_X_gb = X.nbytes / (1024 ** 3)
        size_y_gb = y.nbytes / (1024 ** 3)
        total_gb = size_X_gb + size_y_gb
        perc_ram = (total_gb / total_ram_gb) * 100
        y_flat = np.ravel(y)

        print(f"Set de {nombre}:")
        print(f"\t{n_samples} ventanas")
        print(f"\tTamaño X: {size_X_gb:.3f} GB")
        print(f"\tTamaño y: {size_y_gb:.6f} GB")
        print(f"\tTOTAL: {total_gb:.3f} GB → {perc_ram:.1f}% RAM\n")

    # Mostrar info solo si silent=False
    print_set_info("entrenamiento", X_train, y_train)
    print_set_info("validación",    X_valid, y_valid)
    print_set_info("testeo",        X_test,  y_test)

    # Pesos = cantidad de ventanas
    w_train = X_train.shape[0]
    w_valid = X_valid.shape[0]
    w_test  = X_test.shape[0]

    return w_train, w_valid, w_test

### 2.1 Carga de ventanas 90 minutos

In [17]:
ram_usage()

RAM total:      52.96 GB
RAM usada:      1.80 GB
RAM disponible: 50.54 GB


In [18]:
# Diccionarios para almacenar datos escalados por fold
X_train_sc = {}
y_train_sc = {}
X_valid_sc = {}
y_valid_sc = {}
X_test_sc  = {}
y_test_sc  = {}
scalers    = {}

for k in k_folds:
    print(f'Fold {k}:')

    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler = load_windows_and_scaler(k)

    # Guardar todo en diccionarios
    X_train_sc[k] = X_train
    y_train_sc[k] = y_train

    X_valid_sc[k] = X_valid
    y_valid_sc[k] = y_valid

    X_test_sc[k]  = X_test
    y_test_sc[k]  = y_test

    scalers[k] = scaler

    print(f"  - Datos escalados cargados y almacenados en diccionarios.")
    print("-" * 40)

Fold 1:
	X_train_sc_1 e y_train_1 extraídos correctamente
	X_valid_sc_1 e y_valid_1 extraídos correctamente
	X_test_sc_1 e y_test_1 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 2:
	X_train_sc_2 e y_train_2 extraídos correctamente
	X_valid_sc_2 e y_valid_2 extraídos correctamente
	X_test_sc_2 e y_test_2 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 3:
	X_train_sc_3 e y_train_3 extraídos correctamente
	X_valid_sc_3 e y_valid_3 extraídos correctamente
	X_test_sc_3 e y_test_3 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 4:
	X_train_sc_4 e y_train_4 extraídos correctamente
	X_valid_sc_4 e y_valid_4 extraídos correctamente
	X_test_sc_4 e y_test_4 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
-------------

In [19]:
pesos_folds = {}

for k in k_folds:
    w_train, w_valid, w_test = xy_info(
        k,
        X_train_sc[k],
        y_train_sc[k],
        X_valid_sc[k],
        y_valid_sc[k],
        X_test_sc[k],
        y_test_sc[k],
        silent=True   # evita imprimir
    )

    pesos_folds[k] = {
        "w_train": w_train,
        "w_valid": w_valid,
        "w_test":  w_test,
    }


In [20]:
pesos_folds

{1: {'w_train': 124279, 'w_valid': 24898, 'w_test': 27852},
 2: {'w_train': 149177, 'w_valid': 24898, 'w_test': 27852},
 3: {'w_train': 174075, 'w_valid': 24898, 'w_test': 27852},
 4: {'w_train': 198973, 'w_valid': 24898, 'w_test': 27852},
 5: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852}}

## **3. Dataset de Métricas**

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [21]:
def load_metrics(subcarpeta: str, data: str):
    data_path = f'{drive_path}/6_tft_model/{subcarpeta}/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [22]:
def metrics_verify(subcarpeta: str, data: str) -> bool:
    data_path = f'{drive_path}/6_tft_model/{subcarpeta}/{data}.parquet'
    return os.path.exists(data_path)



In [23]:
def load_or_create_metrics (subcarpeta: str, data:str):
  if metrics_verify(subcarpeta, data):
      print(f"Las métricas existen y son almacenadas en {data[2:len(data)]}")
      model_metrics = load_metrics(subcarpeta, data)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[2:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [24]:
baseline_folds_metrics, flag_baseline_folds_metrics = load_or_create_metrics("6_1_baseline_model", "0_baseline_folds_metrics")
baseline_metrics, flag_baseline_metrics = load_or_create_metrics("6_1_baseline_model", "1_baseline_metrics")

Las métricas existen y son almacenadas en baseline_folds_metrics
Las métricas existen y son almacenadas en baseline_metrics


In [25]:
baseline_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
TFT_baseline_fold_1,0.003212,0.003149,0.650613,115.575687,0.748033
TFT_baseline_fold_2,0.004434,0.003275,0.306372,112.206869,0.721970
TFT_baseline_fold_3,0.001724,0.001612,0.882557,86.496184,0.832883
TFT_baseline_fold_4,0.002223,0.002129,0.791015,110.805684,0.726958
TFT_baseline_fold_5,0.001209,0.000955,0.937221,70.778410,0.850363


### 3.2. Función para guardar métricas

In [26]:
def save_metrics (metrics,  subcarpeta: str, metrics_name: str):
  metrics_path = f"{drive_path}/6_tft_model/{subcarpeta}/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [27]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [28]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

### 3.4. Rutas para guardar modelos por fold

In [29]:
import os

# --- Función unificada ---
def ruta_modelo_fold(fold: int, subcarpeta: str) -> dict:
    """
    Crea la carpeta destino y devuelve la ruta completa
    para almacenar el modelo correspondiente al fold.
    """
    base = f"{drive_path}/6_tft_model/{subcarpeta}"
    os.makedirs(base, exist_ok=True)

    model_path = os.path.join(base, f"baseline_fold_{fold}.pt")
    return {"model_path": model_path}

## **4. Re-formateo más Encoder mínimo**

### **4.1. Helper: de 2D (aplanado) a 3D (B, T, F)**

Este bloque define una función auxiliar utilizada para **re-formatear las ventanas de datos** desde una representación 2D a la representación 3D requerida por el modelo.

- **Entrada**:
  - `X_flat`: matriz 2D con forma *(N, window_size × n_features)*.
  - `window_size`: longitud temporal de la ventana.
  - `n_features`: cantidad de features por paso temporal.

- **Funcionamiento**:
  - Verifica que la entrada sea efectivamente una matriz 2D.
  - Valida la consistencia dimensional comprobando que  
    `window_size × n_features == X_flat.shape[1]`.
  - Reconvierte los datos al formato *(N, window_size, n_features)* mediante `reshape`.

- **Salida**:
  - Un arreglo 3D listo para ser utilizado como entrada del modelo.

Este helper se utiliza para **cada conjunto de datos (train, valid y test)**.  
En este proyecto se emplea `window_size = 90` y `n_features_90 = 12`, asegurando que cada ventana temporal esté correctamente estructurada antes del entrenamiento.

In [30]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

En este bloque se definen los parámetros estructurales de entrada del modelo:

- `window_size = 90`  
  Establece la longitud de la ventana temporal, es decir, la cantidad de minutos consecutivos utilizados como entrada para cada muestra.

- `features_base`  
  Contiene las variables OHLCV básicas del mercado: *open, high, close, low y volume*.

- `features_90`  
  Se construye combinando las variables base con los factores adicionales definidos en `features_to_90`, conformando el conjunto completo de features utilizadas por el modelo.

- `n_features_90`  
  Representa la cantidad total de features por paso temporal y se obtiene como la longitud de `features_90`.

Este bloque permite **verificar explícitamente** el conjunto de features y su cardinalidad, asegurando coherencia dimensional con la configuración del modelo y las funciones de re-formateo de ventanas.

In [31]:
window_size = 90
features_base = ['open','high','close','low','volume']
features_90 = features_base + features_to_90
n_features_90 = len (features_90)
print(f'features_90:\t\t {features_90}')
print(f'n_features_90:\t {n_features_90}')

features_90:		 ['open', 'high', 'close', 'low', 'volume', 'ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']
n_features_90:	 12


**Re-formateo de ventanas y liberación de memoria**

Este bloque se encarga de **transformar las ventanas de entrada desde formato 2D a formato 3D**, de acuerdo con la configuración definida previamente (`window_size = 90` y `n_features_90 = 12`), y de **optimizar el uso de memoria** durante el procesamiento por fold.

- Para cada *fold* en `k_folds`:
  - Las matrices escaladas `X_train_sc[k]`, `X_valid_sc[k]` y `X_test_sc[k]`, originalmente en formato  
    *(N, window_size × n_features_90)*, se convierten al formato requerido por el modelo:  
    *(N, window_size, n_features_90)* mediante la función `reshape_windows`.
  - Los datos re-formateados se almacenan en los diccionarios `Xtr`, `Xva` y `Xte`, indexados por fold.

- Una vez completado el re-formateo de cada fold:
  - Se eliminan explícitamente las matrices 2D originales para evitar duplicación innecesaria de datos en memoria.
  - Se invoca el recolector de basura (`gc.collect()`) para liberar RAM de forma inmediata.

El objetivo principal es asegurar que cada conjunto de datos esté correctamente estructurado para el entrenamiento del modelo Transformer, manteniendo un consumo de memoria controlado durante el procesamiento de múltiples folds.

In [32]:
import gc

Xtr = {}
Xva = {}
Xte = {}

for k in k_folds:
    Xtr[k] = reshape_windows(X_train_sc[k], window_size, n_features_90)
    Xva[k] = reshape_windows(X_valid_sc[k], window_size, n_features_90)
    Xte[k] = reshape_windows(X_test_sc[k],  window_size, n_features_90)

    # Liberar las matrices 2D de este fold
    del X_train_sc[k], X_valid_sc[k], X_test_sc[k]
    gc.collect()

    print(f'Fold {k} re-shape completo y 2D liberado')

Fold 1 re-shape completo y 2D liberado
Fold 2 re-shape completo y 2D liberado
Fold 3 re-shape completo y 2D liberado
Fold 4 re-shape completo y 2D liberado
Fold 5 re-shape completo y 2D liberado


In [33]:
ytr = y_train_sc.copy()
yva = y_valid_sc.copy()
yte = y_test_sc.copy()

**Verificación de dimensiones de entrada por fold**

Este bloque define una función auxiliar destinada a **verificar las dimensiones de los datos de entrada** del modelo para cada fold.

- La función itera sobre los *folds* definidos en `k_folds`.
- Para cada fold:
  - Muestra de forma ordenada los *shapes* de los conjuntos **train**, **valid** y **test**.
  - Verifica que cada conjunto se encuentre en formato 3D, consistente con la estructura  
    *(n_samples, window_size, n_features)* requerida por el modelo.

- La salida se presenta en forma tabular, facilitando la inspección visual y la detección temprana de inconsistencias dimensionales entre folds o conjuntos de datos.

El objetivo principal es confirmar que el re-formateo de las ventanas se haya realizado correctamente antes de proceder al entrenamiento y evaluación del modelo.


In [34]:
def mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte):
    for k in k_folds:
        print(f"\nShapes del Fold {k}")
        print(f"{'Set':<10}{'Shape (3D)':<25}")
        print("-" * 40)

        filas = [
            ("Train", Xtr[k].shape),
            ("Valid", Xva[k].shape),
            ("Test",  Xte[k].shape),
        ]

        for nombre, shape_3d in filas:
            print(f"{nombre:<10}{str(shape_3d):<25}")

In [35]:
mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte)


Shapes del Fold 1
Set       Shape (3D)               
----------------------------------------
Train     (124279, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 2
Set       Shape (3D)               
----------------------------------------
Train     (149177, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 3
Set       Shape (3D)               
----------------------------------------
Train     (174075, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 4
Set       Shape (3D)               
----------------------------------------
Train     (198973, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 5
Set       Shape (3D)               
----------------------------------------
Train     (223871, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852

## **5. Conversión de ventanas temporales 3D a DataFrame compatible con TFT**

### **5.1. Descripción funcional**

Es siguiente código convierte un dataset de ventanas temporales en formato 3D en un DataFrame en formato long compatible con Temporal Fusion Transformer (TFT) de pytorch-forecasting.

El objetivo es adaptar los datos al esquema requerido por la librería, donde cada fila representa un paso temporal dentro de una serie.

---

**Estructura de entrada**

El código parte de los siguientes arreglos:

- X con forma (N, T, F)
  - N: número de ventanas o series temporales
  - T: cantidad de pasos temporales por ventana
  - F: número de features por paso temporal
- y con forma (N,)
  - Un valor objetivo por cada ventana

---

**Estructura de salida**

Se construye un DataFrame donde:

- Cada fila corresponde a un paso temporal
- Cada ventana se identifica mediante un group_id
- El índice temporal se representa con time_idx
- El target se repite en todos los pasos de la ventana
- Las features se aplanan en columnas feat_0, feat_1, …, feat_(F-1)

Este formato es exactamente el esperado por pytorch-forecasting para entrenar modelos como Temporal Fusion Transformer.

### **5.2. Código: construcción del DataFrame para TFT**

In [36]:
import pandas as pd
import numpy as np

def build_tft_df(X, y, prefix="feat", group_offset=0):
    """
    Convierte ventanas 3D (N, T, F) a un DataFrame "long" para TFT (pytorch-forecasting).

    X: np.ndarray (N, T, F)
       N = cantidad de ventanas/series
       T = largo temporal (time steps)
       F = cantidad de features por time step

    y: np.ndarray (N,)
       Target por ventana (un valor por cada serie)

    prefix: str
       Prefijo para las columnas de features: feat_0, feat_1, ...

    group_offset: int
       Offset para que los group_id no se repitan al concatenar folds.
       (Ej: fold1 usa group_id 0..N1-1, fold2 usa N1..N1+N2-1, etc.)
    """

    # Extraemos dimensiones
    N, T, F = X.shape

    # Armamos columnas base del DataFrame:
    # - time_idx: 0..T-1, repetido N veces (una secuencia por ventana)
    # - group_id: id de la ventana/serie, repetido T veces (una por time step)
    # - target: el mismo y de la ventana repetido T veces (nota: ver comentario sobre leakage)
    data = {
        "time_idx": np.tile(np.arange(T), N),
        "group_id": np.repeat(np.arange(N) + group_offset, T),
        "target":   np.repeat(y, T),
    }

    # Aplanamos las features:
    # Para cada feature j, tomamos X[:, :, j] con forma (N, T)
    # y lo "aplanamos" a (N*T,) para que quede alineado con time_idx y group_id
    for j in range(F):
        data[f"{prefix}_{j}"] = X[:, :, j].reshape(-1)

    # Construimos el DataFrame final
    return pd.DataFrame(data)


### **5.3. Generación de DataFrames de entrenamiento y validación para TFT**

A partir de las ventanas temporales 3D y sus targets asociados, se construyen los DataFrames en formato long requeridos por pytorch-forecasting para entrenar un Temporal Fusion Transformer (TFT).

En este paso se generan los conjuntos de entrenamiento y validación, manteniendo el mismo prefijo de features para asegurar consistencia entre ambos datasets.

In [37]:
# Diccionarios para guardar el DataFrame de train y valid por fold
df_tr, df_va = {}, {}

for k in k_folds:
    # Construimos el DataFrame de entrenamiento para el fold k
    df_tr[k] = build_tft_df(Xtr[k], ytr[k], prefix="feat")

    # Construimos el DataFrame de validación para el fold k
    df_va[k] = build_tft_df(Xva[k], yva[k], prefix="feat")

In [38]:
df_te = {}

for k in k_folds:
     # Construimos el DataFrame de testing para el fold k
    df_te[k] = build_tft_df(Xte[k], yte[k], prefix="feat")

Ambos DataFrames comparten la misma estructura (time_idx, group_id, target y features f90_*), condición necesaria para el correcto entrenamiento del modelo TFT.

## **6. Definición de variables para `TimeSeriesDataSet`**

En este paso se definen las listas de variables requeridas por TimeSeriesDataSet de pytorch-forecasting, separando las features temporales y el target según su naturaleza.

Para el horizonte de 90 minutos, todas las features se consideran variables reales conocidas en el tiempo, mientras que el target se define como variable real desconocida.

In [39]:
# Tomamos un fold de referencia (cualquiera)
ref_df = df_tr[k_folds[0]]

feature_cols = [c for c in ref_df.columns if c.startswith("feat_")]

time_varying_known_reals   = feature_cols
time_varying_unknown_reals = ["target"]

static_reals        = []
static_categoricals = []

- `time_varying_known_reals_90`: features disponibles en todos los pasos temporales
- `time_varying_unknown_reals_90`: variable objetivo a predecir
- `static_reals_90 / static_categoricals_90`: no se utilizan variables estáticas en este setup

Estas listas se utilizarán directamente para instanciar el objeto TimeSeriesDataSet del modelo TFT.

## **7. Creación del `TimeSeriesDataSet` para TFT (horizonte 90 minutos)**

En este paso se instancia el objeto TimeSeriesDataSet de pytorch-forecasting, que define cómo el Temporal Fusion Transformer (TFT) consume las series temporales durante el entrenamiento.

Se utiliza un historial de 90 minutos como encoder y se define un horizonte de predicción de 1 paso, correspondiente al retorno a 90 minutos.

### **7.1. Configuración de longitudes temporal**

Nuestras ventanas tienen 90 pasos temporales, y en pytorch-forecasting:
  - `max_encoder_length` = cantidad de pasos usados como entrada
  - `max_prediction_length` = cantidad de pasos a predecir

In [40]:
max_encoder_length = 89      # historial temporal (90 minutos)
max_prediction_length = 1   # retorno a 90 min como un único valor

el modelo ve:
- 89 pasos como historial (encoder)
- 1 paso como predicción (decoder)

- Total efectivo: 90 pasos

Esto es coherente si:
- El target representa el retorno a 90 minutos
- Y ese retorno se asocia al último paso temporal de la ventana

#### **Algunas aclaraciones:**

El modelo utiliza 89 pasos temporales como historial (encoder) y 1 solo paso para la predicción (decoder). Dado que el target está repetido en todos los time_idx dentro de cada ventana, el Temporal Fusion Transformer (TFT) aprende a:

**Predecir el valor del target asociado al último paso temporal de la ventana, es decir, el valor correspondiente a la última fila de cada serie.**

**Cómo construye la muestra el `TimeSeriesDataSet`**

- El encoder utiliza los pasos `time_idx = 0 … 88`
- El decoder contiene un único paso: `time_idx = 89`
- El valor del target utilizado para el entrenamiento corresponde a ese paso del decoder

Como el `target` es constante dentro de la ventana, esto es equivalente a entrenar con el target de la última fila.

**Relación con el problema planteado**

Cada ventana representa:
- 90 minutos de información histórica
- Un único valor objetivo (retorno a 90 minutos)

El modelo aprende la siguiente relación: “Dado el historial completo de la ventana, predecir el retorno futuro asociado a esa ventana”.

**Nota técnica:**

Repetir el target en todos los pasos temporales:
- No afecta negativamente el entrenamiento
- Aunque no es estrictamente necesario

En formulaciones más canónicas, el `target` podría definirse solo en el último `time_idx`.

Para este caso, la implementación utilizada es correcta, consistente y funcional.

### **7.2. Creación del TimeSeriesDataSet**

Para entrenamiento:

In [41]:
# Diccionarios para almacenar los TimeSeriesDataSet
# Uno de entrenamiento y otro de validación por cada fold
training = {}
validation = {}

for k in k_folds:
    # -------------------------------------------------
    # Dataset de ENTRENAMIENTO para el fold k
    # -------------------------------------------------
    training[k] = TimeSeriesDataSet(
        df_tr[k],                 # DataFrame long con las series de entrenamiento del fold k
        time_idx="time_idx",      # Índice temporal dentro de cada serie (0..T-1)
        target="target",          # Variable objetivo a predecir
        group_ids=["group_id"],   # Identificador de cada serie temporal (una por ventana)

        # Longitud del encoder (historial usado por el modelo)
        max_encoder_length=max_encoder_length,

        # Horizonte de predicción (en su caso: 1 paso, retorno a 90 min)
        max_prediction_length=max_prediction_length,

        # Variables reales conocidas en el tiempo (features disponibles en cada paso)
        time_varying_known_reals=time_varying_known_reals,

        # Variables reales desconocidas en el tiempo (incluye el target)
        time_varying_unknown_reals=time_varying_unknown_reals,

        # No se utilizan variables reales estáticas en este setup
        static_reals=static_reals,

        # No se utilizan variables categóricas estáticas en este setup
        static_categoricals=static_categoricals,

        # Se desactiva la normalización automática del target
        # (el escalado se controla externamente en su pipeline)
        target_normalizer=None,
    )

    # -------------------------------------------------
    # Dataset de VALIDACIÓN para el fold k
    # -------------------------------------------------
    # Se construye a partir del dataset de entrenamiento
    # para garantizar misma definición interna (variables, longitudes, etc.)
    validation[k] = TimeSeriesDataSet.from_dataset(
        training[k],   # Dataset de referencia (train del mismo fold)
        df_va[k],         # DataFrame long de validación
        predict=True,     # Indica que este dataset se usa para predicción/evaluación
        stop_randomization=True,  # Desactiva aleatoriedad (reproducibilidad)
    )

In [42]:
test = {}

for k in k_folds:
    # -------------------------------------------------
    # Dataset de TEST para el fold k
    # -------------------------------------------------
    # Se construye a partir del training[k] para heredar:
    # - mismas variables
    # - mismas longitudes (encoder/prediction)
    # - misma configuración interna
    test[k] = TimeSeriesDataSet.from_dataset(
        training[k],        # Dataset base (train del mismo fold)
        df_te[k],           # DataFrame long de test del fold k
        predict=True,       # modo predicción/evaluación
        stop_randomization=True,
    )

Este dataset define:
- La estructura temporal de las series (time_idx, group_id)
- El tamaño del encoder y del horizonte de predicción
- La separación entre variables conocidas, desconocidas y estáticas
- La ausencia de normalización automática del target (control externo)

El objeto resultante será utilizado directamente para crear los DataLoader y entrenar el modelo TFT.

Notas clave:

- ` from_dataset` garantiza que train y validación tengan exactamente la misma estructura.
- `stop_randomization=True` es fundamental para métricas estables y reproducibles.
- `predict=False` asegura que el target esté disponible para evaluación.

Este bloque es el patrón correcto y recomendado para validación en TFT.

## **8.Creación de `DataLoaders` para entrenamiento y validación**

En este paso se generan los `DataLoaders` a partir de los objetos `TimeSeriesDataSet`, que serán utilizados por el modelo TFT durante el entrenamiento y la validación.

Se define un `batch_size` y se configura:
- Entrenamiento con `shuffle=True` para mezclar las series y mejorar la generalización.
- Validación con `shuffle=False` para mantener una evaluación determinística y reproducible.

In [43]:
# Tamaño de batch para entrenamiento y validación
batch_size = 256

# Diccionarios para almacenar los DataLoader por fold
train_dataloader = {}
val_dataloader   = {}
test_dataloader = {}

for k in k_folds:
    # -------------------------------------------------
    # DataLoader de ENTRENAMIENTO para el fold k
    # -------------------------------------------------
    train_dataloader[k] = training[k].to_dataloader(
        batch_size=batch_size,  # Cantidad de muestras por batch
        shuffle=True,           # Se mezclan las series en entrenamiento
        num_workers=0,          # 0 = seguro en notebooks (evita problemas de multiprocessing)
    )

    # -------------------------------------------------
    # DataLoader de VALIDACIÓN para el fold k
    # -------------------------------------------------
    val_dataloader[k] = validation[k].to_dataloader(
        batch_size=batch_size,  # Mismo batch_size que en train
        shuffle=False,          # No se mezcla en validación (evaluación determinista)
        num_workers=0,
    )

In [44]:
for k in k_folds:
    # -------------------------------------------------
    # DataLoader de TEST para el fold k
    # -------------------------------------------------
    test_dataloader[k] = test[k].to_dataloader(
        batch_size=batch_size,
        shuffle=False,     # test → sin shuffle
        num_workers=0,
    )


## **9. Definición del modelo Temporal Fusion Transformer (TFT)**

En este paso se configura el modelo Temporal Fusion Transformer a partir del TimeSeriesDataSet de entrenamiento (training_90).
Además, se desactiva cuDNN para evitar posibles inconsistencias o errores con ciertas operaciones en GPU durante el entrenamiento.

### **9.1. Clasificación de Hiperparámetros**



**Hiperparámetros de ARQUITECTURA (estructura del modelo)**

Estos definen la capacidad y forma del TFT:
- `hidden_size`
- `attention_head_size`
- `hidden_continuous_size`
- `lstm_layers`
- `dropout` (regularización, pero afecta arquitectura)



In [45]:
# -----------------------------------------
# Hiperparámetros BASE de arquitectura (TFT)
# -----------------------------------------
baseline_arch_params = {
    "hidden_size": 16,
    "attention_head_size": 2,
    "hidden_continuous_size": 8,
    "lstm_layers": 1,
    "dropout": 0.1,
}

**Hiperparámetros de ENTRENAMIENTO / OPTIMIZACIÓN**

Estos controlan cómo se entrena el modelo:
- `learning_rate`
- `loss`
- `reduce_on_plateau_patience`
- `log_interval`
- `log_val_interval`

In [46]:
# -----------------------------------------
# Hiperparámetros BASE de entrenamiento
# -----------------------------------------
baseline_train_params = {
    "learning_rate": 1e-3,
    "loss": RMSE(),
    "reduce_on_plateau_patience": 3,
    "log_interval": 10,
    "log_val_interval": 1,
}

### **9.2. Función para crear modelos**


In [47]:
from pytorch_forecasting import TemporalFusionTransformer

def build_tft_models_by_fold(
    training_datasets: dict,
    k_folds: list,
    arch_params: dict,
    train_params: dict,
):
    """
    Construye un modelo Temporal Fusion Transformer por fold.

    Parámetros
    ----------
    training_datasets : dict
        Diccionario {fold: TimeSeriesDataSet} con los datasets de entrenamiento.

    k_folds : list
        Lista de identificadores de folds (ej: [0,1,2,3,4]).

    arch_params : dict
        Hiperparámetros de arquitectura del TFT
        (hidden_size, attention_head_size, etc.).

    train_params : dict
        Hiperparámetros de entrenamiento/optimización
        (learning_rate, loss, logging, etc.).

    Retorna
    -------
    models : dict
        Diccionario {fold: TemporalFusionTransformer} con un modelo por fold.
    """

    models = {}

    for k in k_folds:
        # Construimos el modelo TFT usando el dataset del fold k
        models[k] = TemporalFusionTransformer.from_dataset(
            training_datasets[k],  # dataset de entrenamiento del fold
            **arch_params,         # hiperparámetros de arquitectura
            **train_params,        # hiperparámetros de entrenamiento
        )

    return models

### **9.3. Creación de modelos baseline**


In [48]:
import torch
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import RMSE

# -------------------------------------------------
# (Opcional) cuDNN: determinismo vs performance
# -------------------------------------------------
# - enabled=True  -> más rápido en GPU (default habitual)
# - enabled=False -> puede ayudar si ve comportamientos raros/no deterministas
torch.backends.cudnn.enabled = True
print("cuDNN enabled:", torch.backends.cudnn.enabled)

# -------------------------------------------------
# Un modelo TFT por fold (recomendado en K-fold)
# -------------------------------------------------

tft_baseline_model = build_tft_models_by_fold(
    training_datasets=training,
    k_folds=k_folds,
    arch_params=baseline_arch_params,
    train_params=baseline_train_params,
)

cuDNN enabled: True


# ENTRENAMIENTO BASELINE

## **10. Entrenamiento del modelo TFT**

### **Teoría**

En este punto se entrena el modelo Temporal Fusion Transformer (TFT) utilizando PyTorch Lightning. Se plantean dos etapas:

Entrenamiento 1 (corto): corrida rápida para verificar que el pipeline funciona correctamente (datos, modelo, GPU, métricas).

Entrenamiento 2 (largo): entrenamiento completo incorporando Early Stopping y monitoreo del Learning Rate, para detener el entrenamiento cuando la validación deje de mejorar y registrar la evolución del LR.

**Hiperparámetros del ENTRENAMIENTO (Trainer / Lightning)**

Estos NO pertenecen al modelo, sino al loop de entrenamiento: `pl.Trainer(...)`. Estos también deberían estar en un diccionario baseline, para poder tunearlos o modificarlos de forma controlada.

In [49]:
baseline_trainerL_params = {
    "max_epochs": 30,
    "precision": 32,
    "gradient_clip_val": 0.1,
    "enable_checkpointing": False,
    "log_every_n_steps": 10,
}

**Regla mental para no confundirse**

- Modelo (TFT) → `baseline_arch_params`, `baseline_train_params`
- Entrenamiento (Lightning Trainer) → `baseline_trainerL_params`

Son capas distintas y ambas deben ser parametrizables.

### **10.1. Entrenamiento 1 (se omite en esta etapa)**

In [50]:
TRAINING_PIPELINE_PROBE = '''
#Entrenamiento 1

trainer_0 = pl.Trainer(
    max_epochs=5,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    precision=32,             # Mantener 32 bits para estabilidad
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    log_every_n_steps=10,
)

trainer_0.fit(
    tft_90,
    train_dataloaders=train_dataloader_90,
    val_dataloaders=val_dataloader_90,
)
'''

### **10.2. Early Stopping y monitoreo del Learning Rate (para Entrenamiento 2)**

In [51]:
# Early Stopping para segundo entrenamiento

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
import torch

early_stop = EarlyStopping(
    monitor="val_loss",  # métrica de validación (Lightning)
    patience=3,       # épocas sin mejora antes de detener
    mode="min"
)

#Solo monitorea el LR, no afecta entrenamiento.
lr_monitor = LearningRateMonitor(logging_interval="epoch")

### **10.3. Rutas para guardado**

In [52]:
# --- Generar diccionario de rutas ---
subcarpeta = "6_1_baseline_model"
path_tft_baseline = {}

for k in k_folds:
    path_tft_baseline[k] = ruta_modelo_fold(k, subcarpeta)

# --- Resultado final ---
path_tft_baseline

{1: {'model_path': '/content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_1.pt'},
 2: {'model_path': '/content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_2.pt'},
 3: {'model_path': '/content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_3.pt'},
 4: {'model_path': '/content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_4.pt'},
 5: {'model_path': '/content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_5.pt'}}

### **10.4. Entrenamiento 2 (entrenamiento completo con Early Stopping)**

In [53]:
import logging
logging.getLogger("lightning").setLevel(logging.WARNING)
logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("lightning.pytorch.accelerators.cuda").setLevel(logging.WARNING)

In [54]:
PROBE = '''
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

print("baseline_trainerL_params:", baseline_trainerL_params)

for k in k_folds:
    print(k, "len(train_dl)=", len(train_dataloader[k]), "len(val_dl)=", len(val_dataloader[k]))

    '''

In [55]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
import torch
import os

# Diccionario para almacenar el Trainer por fold (opcional)
trainer_1 = {}


for k in k_folds:

    model_path = path_tft_baseline[k]["model_path"]

    # -------------------------------------------------
    # Chequeo previo: ¿ya existe el modelo del fold?
    # -------------------------------------------------
    if os.path.exists(model_path):
        print(f"[Fold {k}] Modelo ya existe → se omite entrenamiento")
        print(f"[Fold {k}] Path: {model_path}")
        continue

    print(f"\n==============================")
    print(f"[Fold {k}] Inicio de entrenamiento")
    print(f"CUDA disponible: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("==============================")

    # EarlyStopping NUEVO por fold (no reutilizar el mismo objeto)
    early_stop_k = EarlyStopping(
        monitor="val_loss",
        patience=3,
        mode="min",
    )
    # Asegurar que el modelo esté en modo entrenamiento
    tft_baseline_model[k].train()

    # -------------------------------------------------
    # Trainer de ENTRENAMIENTO PRINCIPAL para el fold k
    # -------------------------------------------------
    trainer_1[k] = pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        callbacks=[early_stop_k], #, lr_monitor],
        enable_progress_bar=False,   # ← evita tqdm / widgets
        logger=False,                # ← evita logs visuales
        enable_model_summary=False,  # ← evita prints extra
        **baseline_trainerL_params,
    )

    # -------------------------------------------------
    # Entrenamiento del modelo TFT del fold k
    # -------------------------------------------------
    print(f"[Fold {k}] Epoch inicial:", trainer_1[k].current_epoch)

    trainer_1[k].fit(
        tft_baseline_model[k],        # modelo del fold k
        train_dataloaders=train_dataloader[k],  # dataloader train del fold k
        val_dataloaders=val_dataloader[k],      # dataloader valid del fold k
    )

    # -------------------------------------------------
    # Métricas finales
    # -------------------------------------------------

    metrics = trainer_1[k].callback_metrics
    if "train_loss" in metrics:
        print(f"[Fold {k}] train_loss:", metrics["train_loss"].item())

    if "val_loss" in metrics:
        print(f"[Fold {k}] val_loss:", metrics["val_loss"].item())


    # -------------------------------------------------
    # Guardar modelo del fold k
    # -------------------------------------------------

    model_path = path_tft_baseline[k]["model_path"]
    os.makedirs(os.path.dirname(model_path), exist_ok=True)

    torch.save(
        tft_baseline_model[k].state_dict(),
        model_path
    )
    print(f"[Fold {k}] Modelo guardado en: {model_path}")

    # -------------------------------------------------
    # Información de control (DEBUG / diagnóstico)
    # -------------------------------------------------
    print(f"[Fold {k}] Epoch final:", trainer_1[k].current_epoch)
    print(f"[Fold {k}] Épocas completadas:", trainer_1[k].fit_loop.epoch_progress.current.completed)
    print(f"[Fold {k}] EarlyStopping activado:", trainer_1[k].should_stop)

[Fold 1] Modelo ya existe → se omite entrenamiento
[Fold 1] Path: /content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_1.pt
[Fold 2] Modelo ya existe → se omite entrenamiento
[Fold 2] Path: /content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_2.pt
[Fold 3] Modelo ya existe → se omite entrenamiento
[Fold 3] Path: /content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_3.pt
[Fold 4] Modelo ya existe → se omite entrenamiento
[Fold 4] Path: /content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_4.pt
[Fold 5] Modelo ya existe → se omite entrenamiento
[Fold 5] Path: /content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/baseline_fold_5.pt


# TUNING GRUESO

##**11. Coarse Tuning TFT**

In [56]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.6 MB/s eta 0:00:00


In [62]:
max_encoder_length

89

### **11.1. Código `suggest_hparams_coarse(trial)`**

#### **11.1.1. Descripción**

Este bloque define la función de sugerencia de hiperparámetros para el tuneo grueso (coarse tuning) del modelo Temporal Fusion Transformer (TFT) usando Optuna.

El objetivo es explorar arquitecturas y parámetros de entrenamiento de forma eficiente, evitando combinaciones inválidas.

---

**Idea clave: restricción entre `hidden_size` y `attention_head_size`**

En modelos con atención, el número de cabezas debe ser compatible con el tamaño interno del modelo.
Para garantizarlo, se define explícitamente una restricción:

- `hidden_size` = 16 → heads ∈ {1, 2, 4}
- `hidden_size` = 32 → heads ∈ {1, 2, 4, 8}
- `hidden_size` = 64 → heads ∈ {2, 4, 8}

Optuna **solo puede proponer combinaciones válidas**, evitando trials inútiles o inestables.

---

**¿Qué hiperparámetros se tunean en esta etapa?**

Arquitectura del modelo (capacidad):

- `hidden_size`
- `attention_head_size` (con restricción)
- `hidden_continuous_size`
- `lstm_layers`
- `dropout`

Entrenamiento (optimización y estabilidad):

- `learning_rate`
- `gradient_clip_val`

---

**¿Por qué es coarse tuning?**

- Explora órdenes de magnitud, no valores finos
- Usa un espacio amplio pero controlado
- Sirve para descartar regiones malas del espacio
- Prepara el terreno para un fine tuning posterior

---

**Resultado**

La función suggest_hparams devuelve un diccionario con hiperparámetros consistentes y válidos, listo para:

- Construir el modelo TFT
- Entrenarlo rápidamente
- Evaluar su desempeño en validación

Este enfoque hace que el tuneo sea más rápido, estable y reproducible.


#### **11.1.2. Código `suggest_hparams_coarse(trial)`**

In [57]:
# ============================================================
# Coarse Tuning TFT — Restricción heads vs hidden_size (Optuna)
# ============================================================

import optuna
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
import torch

from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import RMSE

# ------------------------------------------------------------
# Restricción explícita entre hidden_size y attention_head_size
# ------------------------------------------------------------
# Define qué valores de attention_head_size son válidos
# para cada hidden_size. Esto evita combinaciones inválidas
# o poco razonables durante el coarse tuning.
ALLOWED_HEADS_BY_HIDDEN = {
    16: [1, 2, 4],
    32: [1, 2, 4, 8],
    64: [2, 4, 8],
}

def suggest_hparams(trial: optuna.Trial):
    """
    Sugiere hiperparámetros para el COARSE TUNING del TFT,
    garantizando que attention_head_size sea compatible
    con hidden_size.
    """
    hp = {}

    # -------------------------
    # 1) Arquitectura del modelo
    # -------------------------

    # Tamaño del estado oculto principal del TFT
    hidden_size = trial.suggest_categorical("hidden_size", [16, 32, 64])
    hp["hidden_size"] = hidden_size

    # Restricción clave:
    # el número de cabezas de atención se elige SOLO
    # entre las permitidas para ese hidden_size
    allowed_heads = ALLOWED_HEADS_BY_HIDDEN[hidden_size]
    hp["attention_head_size"] = trial.suggest_categorical(
        "attention_head_size",
        allowed_heads
    )

    # Tamaño de la proyección interna de variables continuas
    hp["hidden_continuous_size"] = trial.suggest_categorical(
        "hidden_continuous_size", [8, 16, 32]
    )

    # Número de capas LSTM internas del TFT
    hp["lstm_layers"] = trial.suggest_categorical("lstm_layers", [1, 2])

    # Dropout para regularización
    hp["dropout"] = trial.suggest_categorical(
        "dropout", [0.05, 0.10, 0.20, 0.30]
    )

    # -------------------------
    # 2) Hiperparámetros de training (modelo)
    # -------------------------

    # Learning rate del optimizador
    hp["learning_rate"] = trial.suggest_categorical(
        "learning_rate", [1e-4, 3e-4, 1e-3]
    )

    # -------------------------
    # 3) Hiperparámetros de training (Trainer)
    # -------------------------

    # Gradient clipping para estabilidad del entrenamiento
    hp["gradient_clip_val"] = trial.suggest_categorical(
        "gradient_clip_val", [0.05, 0.10, 0.20, 0.50]
    )

    return hp


### **11.2. Código `merge_params`**

#### **11.2.1. Descripción funcional**

La siguiente función fusiona los hiperparámetros baseline del modelo TFT con los hiperparámetros sugeridos durante el coarse tuning, manteniendo una separación clara de responsabilidades.

El objetivo es **evitar mezclar parámetros del modelo con parámetros del Trainer**, algo fundamental en PyTorch Lightning.

---

**Tres diccionarios distintos**

En TFT (y en Lightning) hay tres niveles diferentes de configuración:

1. Arquitectura del modelo
    - Parámetros que definen la capacidad del TFT
    - Ejemplos: `hidden_size`, `attention_head_size`, `dropout`

2. Entrenamiento del modelo
    - Parámetros internos del modelo relacionados con la optimización
    - Ejemplos: `learning_rate`, `loss`, `reduce_on_plateau_patience`

3. Entrenamiento a nivel Trainer
    - Parámetros que controlan el loop de entrenamiento
    - Ejemplo: `gradient_clip_val`

Esta función garantiza que cada parámetro vaya al lugar correcto.

---

**Funcionamiento interno**

- Parte de los diccionarios baseline
- Recorre los parámetros sugeridos por Optuna
- Según el nombre del parámetro:
  - Lo asigna a arch_params
  - o a train_params
  - o a trainer_params

De esta forma:

  - No se pierden valores baseline
  - Solo se sobrescriben los parámetros que se están tuneando
  - El código queda limpio y reutilizable

---

**Resultado**

La función devuelve:

- `arch_params` → se pasan a `TemporalFusionTransformer.from_dataset(...)`
- `train_params` → se pasan a `TemporalFusionTransformer.from_dataset(...)`
- `trainer_params` → se pasan a `pl.Trainer(...)`

Esto nos permitirá tener un coarse tuning ordenado, seguro y reproducible, alineado con las buenas prácticas que usamos en el coarse tuning del Transformer.

#### **11.2.2. Código `merge_params`**

In [58]:
def merge_params(base_arch: dict, base_train: dict, suggested: dict):
    """
    Combina los hiperparámetros baseline con los sugeridos por Optuna.

    Devuelve tres diccionarios separados:
      - arch_params: parámetros de arquitectura del modelo (para from_dataset)
      - train_params: parámetros de entrenamiento del modelo (para from_dataset)
      - trainer_params: parámetros del Trainer (pl.Trainer)
    """

    # Copia de los hiperparámetros base para no modificarlos in-place
    arch = dict(base_arch)     # arquitectura del TFT
    train = dict(base_train)   # entrenamiento del modelo
    trainer = {}               # entrenamiento a nivel Trainer

    # Recorre los hiperparámetros sugeridos por Optuna
    for k, v in suggested.items():

        # Si el parámetro pertenece a la arquitectura del modelo
        if k in arch:
            arch[k] = v

        # Si el parámetro pertenece al entrenamiento del modelo
        elif k in train:
            train[k] = v

        # Si el parámetro es propio del Trainer (no del modelo)
        elif k == "gradient_clip_val":
            trainer["gradient_clip_val"] = v

    # Devuelve los tres diccionarios separados y listos para usar
    return arch, train, trainer


### **11.3. Código `objective(trial)`**

####**11.3.1. Descripción funcional**

Este bloque define la función objective de Optuna para realizar el tuneo grueso (coarse tuning) del modelo Temporal Fusion Transformer (TFT).

Cada vez que Optuna ejecuta un trial, esta función:

1. Propone un conjunto de hiperparámetros
2. Construye un modelo TFT con esos parámetros
3. Lo entrena durante pocas épocas
4. Evalúa su desempeño en validación
5. Devuelve una métrica para comparar trials

---

**Usamos un solo fold**

En coarse tuning el objetivo es:

- Explorar rápidamente el espacio de hiperparámetros
- Descartar configuraciones malas
- Reducir el costo computacional

Usar un solo fold y pocas épocas permite evaluar muchas configuraciones sin gastar horas de cómputo.

---

**Métrica a optimizar**

 - Se optimiza  `val_loss`
- En este setup, `val_loss` corresponde a RMSE
- Optuna busca minimizar este valor

Si un trial no produce `val_loss`, se penaliza devolviendo infinito.

---

**Early Stopping agresivo**

- `patience=2`
- pocas épocas (`max_epochs=12`)

Esto permite:

- Cortar rápidamente modelos que no mejoran
- Ahorrar tiempo
- Enfocar el cómputo en configuraciones prometedoras

---

**Resultado**

La función `objective` devuelve un único valor numérico por trial, que Optuna usa para:

- Comparar configuraciones
- Seleccionar las mejores regiones del espacio
- Definir qué hiperparámetros pasarán al fine tuning

Este enfoque hace que el coarse tuning del TFT sea rápido, controlado y reproducible, alineado con la estrategia que ya usamos para el Transformer.

####**11.3.2. Código `objective(trial)`**

In [74]:
import numpy as np
import gc
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from pytorch_forecasting import TemporalFusionTransformer

def objective(trial):

    # -------------------------
    # Info de inicio de trial
    # -------------------------
    print(f"\n==============================")
    print(f"[Trial {trial.number}] INICIO")
    print("==============================")

    # 1) HPs sugeridos (coarse, con restricción)
    hp = suggest_hparams(trial)

    # 2) Separar baseline + suggested
    arch_p, train_p, trainer_p = merge_params(
        baseline_arch_params,
        baseline_train_params,
        hp
    )

    fold_val_losses = []

    # -------------------------
    # Loop por folds
    # -------------------------
    for i, k in enumerate(k_folds, start=1):
        print(f"[Trial {trial.number}] → Entrenando Fold {k} ({i}/{len(k_folds)})")

        # Dataset del fold k (TimeSeriesDataSet)
        ds_k = training[k]

        # Construir modelo del fold
        # Quitar "loss" de los kwargs para evitar deepcopy en save_hyperparameters
        train_p_safe = dict(train_p)
        train_p_safe.pop("loss", None)

        model_k = TemporalFusionTransformer.from_dataset(
            ds_k,
            **arch_p,
            output_size=1,   # <- fuerza salida 1
        )

        # Setear el loss después (ya no pasa por save_hyperparameters)
        model_k.loss = RMSE()


        # EarlyStopping por fold
        early_stop_k = EarlyStopping(
            monitor="val_loss",
            patience=2,
            mode="min"
        )

        # Trainer = baseline + overrides coarse + trial
        coarse_trainerL_params = dict(baseline_trainerL_params)
        coarse_trainerL_params.update({
            "max_epochs": 12,
            "logger": False,
            "enable_progress_bar": False,
            "enable_model_summary": False,
            "enable_checkpointing": False,
            "log_every_n_steps": 10,
            "callbacks": [early_stop_k],
            "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
            "precision": 32,
        })
        coarse_trainerL_params.update(trainer_p)

        trainer = pl.Trainer(**coarse_trainerL_params)

        # Entrenamiento
        trainer.fit(
            model_k,
            train_dataloaders=train_dataloader[k],
            val_dataloaders=val_dataloader[k],
        )

        # Extraer val_loss
        val_loss_k = trainer.callback_metrics.get("val_loss", None)
        if val_loss_k is None:
            print(f"[Trial {trial.number} | Fold {k}] ❌ val_loss no disponible")
            return float("inf")

        val_loss_k = float(val_loss_k.detach().cpu())
        fold_val_losses.append(val_loss_k)

        print(f"[Trial {trial.number} | Fold {k}] ✔ val_loss = {val_loss_k:.6f}")

        # Limpieza
        del trainer
        del model_k
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -------------------------
    # Resultado final del trial
    # -------------------------
    mean_val = float(np.mean(fold_val_losses))

    print(f"[Trial {trial.number}] FIN → mean val_loss = {mean_val:.6f}")

    return mean_val



### **11.4. Guardado de estudio**

En esta etapa se definen y utilizan rutas de salida para guardar:

**Del estudio de Optuna (una sola vez):**
- `study.pkl`: objeto completo del estudio (permite reanudar o auditar).
- `best_params.json`: mejores hiperparámetros encontrados.
- `trials.csv`: tabla con todos los trials y su score/métricas.

**Resultados finales:**
- `checkpoint_fold_k.pt`: checkpoint por fold (pesos + scaler_y + métricas + hparams).
- `final_metrics.csv`: resumen final (una fila por fold + promedios).
- *(opcional)* `notes.txt`: fecha, dataset tag, features, horizonte, etc.

In [75]:
import os, json
import pandas as pd

# -------------------------------------------------
# 1) Carpeta base del coarse tuning
# -------------------------------------------------
# Se define la ruta donde se almacenarán TODOS los
# artefactos del tuneo grueso (Optuna + resúmenes).
# La carpeta se crea si no existe.
drive_path_coarse_tuning = drive_path + "/6_tft_model/6_2_coarse_tuning"
base_tuned = drive_path_coarse_tuning
os.makedirs(base_tuned, exist_ok=True)

# -------------------------------------------------
# 2) Subcarpeta específica para Optuna
# -------------------------------------------------
# Aquí se guardan:
# - el estudio completo (pickle)
# - los mejores hiperparámetros (JSON)
# - el historial de trials (CSV)
optuna_dir = os.path.join(base_tuned, "optuna_study")
os.makedirs(optuna_dir, exist_ok=True)

# Diccionario con todas las rutas relevantes del estudio
paths_study = {
    # Estudio completo de Optuna (permite reanudar o analizar luego)
    "study_pkl":  os.path.join(optuna_dir, "coarse_study.pkl"),

    # Mejores hiperparámetros encontrados (para reentrenar)
    "best_json":  os.path.join(optuna_dir, "coarse_best_params.json"),

    # Tabla con todos los trials ejecutados
    "trials_csv": os.path.join(optuna_dir, "coarse_trials.csv"),
}

# -------------------------------------------------
# 3) Ruta para métricas finales del coarse tuning
# -------------------------------------------------
# Se usará luego para guardar un resumen final
# (por ejemplo: RMSE, DirAcc, etc. del mejor modelo)
path_final_metrics = os.path.join(base_tuned, "coarse_final_metrics.csv")

# Mostrar rutas creadas (sanity check)
paths_study, path_final_metrics

({'study_pkl': '/content/drive/MyDrive/neural_profit/6_tft_model/6_2_coarse_tuning/optuna_study/coarse_study.pkl',
  'best_json': '/content/drive/MyDrive/neural_profit/6_tft_model/6_2_coarse_tuning/optuna_study/coarse_best_params.json',
  'trials_csv': '/content/drive/MyDrive/neural_profit/6_tft_model/6_2_coarse_tuning/optuna_study/coarse_trials.csv'},
 '/content/drive/MyDrive/neural_profit/6_tft_model/6_2_coarse_tuning/coarse_final_metrics.csv')

### **11.4. Ejecución de `objective(trial)`**

Una vez que tenemos definidass estas piezas:

- `suggest_hparams(...)`
- `merge_params(...)`
- `objective(...)`
- `training`, `train_dataloader`, `val_dataloader`

podemos empezar a crear y ejecutar el estudio de Optuna.

In [ ]:
import optuna

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=25, show_progress_bar=True)

print("Best value (mean val_loss):", study.best_value)
print("Best params:", study.best_params)

[I 2025-12-22 21:02:43,293] A new study created in memory with name: no-name-7e5bd532-a0f4-43ac-9123-40dd7b59d331


  0%|          | 0/25 [00:00<?, ?it/s]


[Trial 0] INICIO
[Trial 0] → Entrenando Fold 1 (1/5)


## **11. Predicciones con el modelo TFT**


En este punto se definen las funciones necesarias para obtener las predicciones del modelo TFT sobre el conjunto de validación.

El objetivo es extraer, de forma controlada y sin gradientes, los valores predichos por el modelo y los valores reales del target, para luego poder evaluar métricas.

### **11.1. Obtener predicciones**

La siguiente función:

- Pone el modelo en modo evaluación (eval)
- Itera sobre el DataLoader de validación
- Extrae las predicciones del output del TFT
- Maneja correctamente los distintos formatos de salida de pytorch-forecasting
- Devuelve y_true y y_pred como arrays de NumPy, listos para evaluación

### **11.1. Evaluar TFT por fold y por split (train/valid/test)**

In [ ]:
def eval_tft_one(model, dataloader):
    """Devuelve (y_true, y_pred) desde un dataloader (sin guardar métricas)."""
    model.eval()
    y_true_list, y_pred_list = [], []

    for batch_x, batch_y in iter(dataloader):
        with torch.no_grad():
            out = model(batch_x)

        # predicciones
        if hasattr(out, "prediction"):
            preds = out.prediction
        elif isinstance(out, dict) and "prediction" in out:
            preds = out["prediction"]
        else:
            preds = out

        # targets
        y = batch_y[0] if isinstance(batch_y, tuple) else batch_y

        y_true_list.append(y)
        y_pred_list.append(preds)

    y_true = torch.cat(y_true_list, dim=0).detach().cpu().numpy().ravel()
    y_pred = torch.cat(y_pred_list, dim=0).detach().cpu().numpy().ravel()
    return y_true, y_pred


def eval_tft_splits_by_fold(models, dl_train, dl_valid, dl_test, k_folds):
    """
    Evalúa train/valid/test por fold.
    Retorna dicts: y_true_split[fold], y_pred_split[fold] para cada split.
    """
    y_true = {"train": {}, "valid": {}, "test": {}}
    y_pred = {"train": {}, "valid": {}, "test": {}}

    for k in k_folds:
        y_true["train"][k], y_pred["train"][k] = eval_tft_one(models[k], dl_train[k])
        y_true["valid"][k], y_pred["valid"][k] = eval_tft_one(models[k], dl_valid[k])
        y_true["test"][k],  y_pred["test"][k]  = eval_tft_one(models[k], dl_test[k])

    return y_true, y_pred


In [ ]:
y_true_splits, y_pred_splits = eval_tft_splits_by_fold(
    models=tft_baseline_model,
    dl_train=train_dataloader,
    dl_valid=val_dataloader,
    dl_test=test_dataloader,
    k_folds=k_folds,
)

### **11.2. Armar DF tipo transformer”: columnas train_, valid_, test_ por fold**

In [ ]:
import pandas as pd

def build_fold_metrics_df_tft(k_folds, y_true, y_pred):
    """
    Construye un DataFrame con una fila por fold y columnas:
    train_RMSE... valid_RMSE... test_RMSE...
    """
    rows = {}

    for k in k_folds:
        row = {}

        # TRAIN
        m_tr = evaluate_model(model=None, X=None, y_true=y_true["train"][k], y_pred=y_pred["train"][k])
        for mname, val in m_tr.items():
            row[f"train_{mname}"] = val

        # VALID
        m_va = evaluate_model(model=None, X=None, y_true=y_true["valid"][k], y_pred=y_pred["valid"][k])
        for mname, val in m_va.items():
            row[f"valid_{mname}"] = val

        # TEST
        m_te = evaluate_model(model=None, X=None, y_true=y_true["test"][k], y_pred=y_pred["test"][k])
        for mname, val in m_te.items():
            row[f"test_{mname}"] = val

        rows[f"TFT_baseline_fold_{k}"] = row

    return pd.DataFrame.from_dict(rows, orient="index")

In [ ]:
baseline_folds_metrics = build_fold_metrics_df_tft(
    k_folds=k_folds,
    y_true=y_true_splits,
    y_pred=y_pred_splits,
)

baseline_folds_metrics

,train_RMSE,train_MAE,train_R2,train_SMAPE,train_DirAcc,valid_RMSE,valid_MAE,valid_R2,valid_SMAPE,valid_DirAcc,test_RMSE,test_MAE,test_R2,test_SMAPE,test_DirAcc
TFT_baseline_fold_1,0.003116,0.003097,0.628113,116.709854,0.746803,0.003107,0.003070,0.722068,101.736412,0.794781,0.003737,0.003457,0.687136,122.886368,0.711733
TFT_baseline_fold_2,0.004321,0.003298,0.321949,111.853638,0.727073,0.003709,0.002956,0.247958,106.973907,0.744805,0.005688,0.003435,0.275157,118.776749,0.674226
TFT_baseline_fold_3,0.001668,0.001651,0.893879,86.836502,0.831398,0.001579,0.001571,0.793896,92.481079,0.822648,0.002205,0.001404,0.891049,79.019066,0.851309
TFT_baseline_fold_4,0.002126,0.002119,0.815029,108.996880,0.734295,0.002137,0.002133,0.589055,127.552666,0.660197,0.002988,0.002198,0.799997,108.756889,0.734230
TFT_baseline_fold_5,0.001080,0.000939,0.949236,69.420319,0.854611,0.001075,0.000962,0.899688,82.473282,0.815037,0.002370,0.001078,0.874195,71.240082,0.847801


#### **Lectura general**

Las métricas muestran un **comportamiento consistente y sano del modelo**, con **heterogeneidad esperable entre folds**. No hay red flags graves.

---

**1. Coherencia train / valid / test (primer bloque)**

En la mayoría de los folds se observa que:

- **Train y valid están cercanos**
- **Test no colapsa** (no hay caída brusca de R² ni explosión de error)

Esto indica que:

- No hay overfitting severo
- El modelo generaliza razonablemente bien

Ejemplos claros:

- **Fold 3**:
  - Train R² ≈ 0.89
  - Valid R² ≈ 0.79
  - Test R² ≈ 0.89

- **Fold 5**:
  - Train R² ≈ 0.95
  - Valid R² ≈ 0.90
  - Test R² ≈ 0.87

Estos folds son **muy sólidos**.

---


**2. Variabilidad entre folds (normal en K-Fold)**

- Fold 2 es el **más débil**:
  - R² bajo (~0.25–0.30)
  - SMAPE alto (>110)

- Fold 3 y 5 son los **mejores**:
  - R² > 0.88
  - SMAPE < 90
  - DirAcc > 0.83

Esto sugiere que:

- El problema es **intrínsecamente no estacionario**
- Algunos períodos son **más predecibles que otros**
- El TFT **no está fallando**, sino reflejando la dificultad del mercado

---

**3. Direccionalidad (muy buen punto)**

- `DirAcc`:
  - la mayoría entre **0.72 y 0.85**
  - test consistente con valid

Esto es **muy positivo** para un setup financiero:

- la dirección se mantiene incluso cuando el error numérico sube
- el modelo captura **señal estructural**, no solo ruido


In [ ]:
if flag_baseline_folds_metrics == False:
  save_metrics(baseline_folds_metrics, "6_1_baseline_model","0_baseline_folds_metrics")
else:
  print("Ya existen métricas del entrenamiento y están guardadas en disco")

Métricas guardadas en /content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/0_baseline_folds_metrics.parquet


### **11.3. Reducir a una sola métrica por fold usando sus pesos (train/valid/test)**

In [ ]:
def weighted_metrics_per_fold(df_split_metrics, fold_weights):
    """
    Genera un DF final con una fila por fold y columnas RMSE/MAE/R2/SMAPE/DirAcc,
    aplicando el promedio ponderado train/valid/test dentro de cada fold.
    """
    metricas = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]
    out = pd.DataFrame(columns=metricas)

    for idx in df_split_metrics.index:
        # Extraer k desde el índice (si lo guarda como ...fold_{k})
        k = int(idx.split("_")[-1])

        w_train = fold_weights[k]["w_train"]
        w_valid = fold_weights[k]["w_valid"]
        w_test  = fold_weights[k]["w_test"]

        row = {}
        for m in metricas:
            col_train = f"train_{m}"
            col_valid = f"valid_{m}"
            col_test  = f"test_{m}"

            row[m] = (
                df_split_metrics.loc[idx, col_train] * w_train +
                df_split_metrics.loc[idx, col_valid] * w_valid +
                df_split_metrics.loc[idx, col_test]  * w_test
            ) / (w_train + w_valid + w_test)

        out.loc[idx] = row

    return out

In [ ]:
baseline_metrics = weighted_metrics_per_fold(
    df_split_metrics=baseline_folds_metrics,
    fold_weights=pesos_folds,
)

baseline_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
TFT_baseline_fold_1,0.003212,0.003149,0.650613,115.575687,0.748033
TFT_baseline_fold_2,0.004434,0.003275,0.306372,112.206869,0.721970
TFT_baseline_fold_3,0.001724,0.001612,0.882557,86.496184,0.832883
TFT_baseline_fold_4,0.002223,0.002129,0.791015,110.805684,0.726958
TFT_baseline_fold_5,0.001209,0.000955,0.937221,70.778410,0.850363


#### **Conclusión**

Las métricas ponderadas son **consistentes**:

- RMSE y MAE similares
- R² alineado
- SMAPE estable
- DirAcc apenas más bajo (esperable al resumir)

No hay contradicción entre ambos reportes.
Entonces:


- El **pipeline es correcto**
- El TFT **aprende señal real**
- No hay overfitting grave
- La variabilidad entre folds es **esperable y realista**
- Fold 3 y 5 confirman el potencial del modelo


In [ ]:
if flag_baseline_metrics == False:
  save_metrics(baseline_metrics,"6_1_baseline_model","1_baseline_metrics")
else:
  print("Ya existen métricas del entrenamiento y están guardadas en disco")

Métricas guardadas en /content/drive/MyDrive/neural_profit/6_tft_model/6_1_baseline_model/1_baseline_metrics.parquet


## **12. Diagnóstico Final**

Este baseline TFT es **válido, competitivo y confiable**.

El siguiente paso **no es arreglar métricas**, sino:

- promediar métricas K-Fold
- comparar contra otros modelos
- o explotar estrategias de ensemble / weighting por régimen

Agregar métricas promedio y desviación estándar K-Fold permitiría cerrar formalmente la evaluación del baseline.